In [ ]:
import sys
import os
# !{sys.executable} -m pip install earthengine-api geemap
sys.path.append(os.path.abspath('..'))


import ee
import geemap
from scripts.aoi_utils import get_gidabo_basin

# Initialize
ee.Initialize(project='ee-my-israelzemedkungebre')
aoi = get_gidabo_basin()

# (Assuming you run the trend logic here or import it)
Map = geemap.Map()
Map.centerObject(aoi, 12)

# Add the Gidabo boundary
Map.addLayer(aoi, {'color': 'red'}, 'Gidabo Basin Boundary')

# Display the map
Map

Map(center=[6.642810173737431, 38.21914863513888], controls=(WidgetControl(options=['position', 'transparent_b…

: 

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import ee, geemap
from scripts.aoi_utils import get_gidabo_basin

# 1. Initialize
ee.Initialize(project='ee-my-israelzemedkungebre')
aoi = get_gidabo_basin()

# 2. Year 2000 Analysis (Landsat 5)
# Using a 3-year window (1999-2001) to ensure we get clear pixels
l5_col = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
    .filterBounds(aoi) \
    .filterDate('1999-01-01', '2001-12-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 30))

# NIR = SR_B4, Red = SR_B3 for Landsat 5
ndvi_2000 = l5_col.median().normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI_2000').clip(aoi)

# 3. Year 2024 Analysis (Landsat 8)
l8_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(aoi) \
    .filterDate('2023-01-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 20))

# NIR = SR_B5, Red = SR_B4 for Landsat 8
ndvi_2024 = l8_col.median().normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_2024').clip(aoi)

# 4. Create the Map
Map = geemap.Map()
Map.centerObject(aoi, 13)

# Visualization: Red (Low Veg) -> Yellow -> Green (Healthy Veg)
vis = {
    'min': 0, 
    'max': 0.7, 
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
}

Map.addLayer(ndvi_2000, vis, 'Vegetation Health 2000')
Map.addLayer(ndvi_2024, vis, 'Vegetation Health 2024')
Map.addLayer(aoi, {'color': 'blue'}, 'Gidabo Boundary', False)

Map

Map(center=[6.642810173737431, 38.21914863513888], controls=(WidgetControl(options=['position', 'transparent_b…